# Stability-constrained unit commitment using NCET

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xuwkk/Neural-SmallSignal-UC-Tutorial/blob/main/stability_constrained_optimization_with_ncet.ipynb)

This notebook contains the complete tutorial workflow:

$$
\text{ANDES data generation}
\rightarrow \text{ReLU neural-network training}
\rightarrow \text{NCET surrogate embedding}
\rightarrow \text{stability-constrained UC}
\rightarrow \text{evaluation}.
$$

For familiarity on the small-signal stability and basic ANDES usage, please refer to [small_signal_concept_with_andes.ipynb](small_signal_concept_with_andes.ipynb).

### Google Colab setup

> **GOOGLE COLAB ONLY:** The next cell clones the complete project and installs the Colab dependencies. It does nothing in a local Jupyter environment.

In [ ]:
# GOOGLE COLAB ONLY: clone the project and install its dependencies.
import os
import subprocess
import sys
from pathlib import Path

IN_GOOGLE_COLAB = "google.colab" in sys.modules
if IN_GOOGLE_COLAB:
    COLAB_REPOSITORY_URL = "https://github.com/xuwkk/Neural-SmallSignal-UC-Tutorial.git"
    COLAB_PROJECT_DIR = Path("/content/Neural-SmallSignal-UC-Tutorial")

    if not COLAB_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", COLAB_REPOSITORY_URL, str(COLAB_PROJECT_DIR)],
            check=True,
        )

    os.chdir(COLAB_PROJECT_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
        check=True,
    )
    print(f"Colab project directory: {COLAB_PROJECT_DIR}")

## 1. Setup

Shared system data are imported from **system_constants.py**, while ANDES operations are imported from **small_signal_functions.py**.

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Import ANDES before PyTorch to avoid a macOS OpenMP runtime conflict.
import system_constants as const
import small_signal_functions as ss
import torch
import neural_network_functions as nnf
from ncet import Bounds, form_milp

display(pd.Series(ss.ENVIRONMENT_INFO, name="value"))

## 2. Generate the ANDES dataset

The input to a small-signal stability assessor is also the operating point of the dynamic simulation.

$$x=[u_2,u_3,u_6,P_{g,2},P_{g,3},P_{g,6},P_s,P_L^{\mathrm{total}}].$$

The all-off commitment is excluded. For each of the other seven commitments, one quarter of the samples broadly cover 0–100 MW solar, while three quarters target 92–96 MW solar with low SG dispatch. This preserves full-range coverage while adding more observations near the stability boundary.

Total active demand is sampled as $P_L^{\mathrm{total}}=s_LP_L^{\mathrm{base}}$ with $s_L\sim U(0.8,1.4)$. The spatial distribution is fixed: $P_{l,i}=s_LP_{l,i}^{\mathrm{base}}$. This matches the UC without network constraints, where only total demand affects the optimization. ANDES still receives every bus load, and each bus retains its base-case Q/P ratio.

Before calling ANDES, candidates are resampled until the approximate lossless Slack output

$$P_{\mathrm{slack}}^{\mathrm{approx}}=P_L^{\mathrm{total}}-\sum_gP_g-P_s$$

is within its limit 50–200 MW. The AC power flow then applies the exact Slack and voltage checks. After dynamic initialization, the real part of the critical eigenvalue, $\alpha$, becomes the regression target. The derived label remains available only as a stability diagnostic.

The notebook uses 100 candidates per commitment; E.g., total 700 candidates (with some of them rejected).

In [ ]:
# Use 100 candidates for each of the seven nonzero commitments.
# Increase this value further for a formal out-of-sample study.
SAMPLES_PER_COMMITMENT = 100
RANDOM_SEED = 7
DATASET_PATH = const.OUTPUT_DIR / "small_signal_dataset.csv"

dataset, failure_counts = ss.generate_dataset(
    samples_per_commitment=SAMPLES_PER_COMMITMENT,   # How many random samples for each commitment status
    random_seed=RANDOM_SEED,
    output_path=DATASET_PATH,
    verbose=False,                # Do not print one line per sample
    show_progress=True,           # Display one progress bar
    suppress_andes_output=True,   # Hide diagnostics from rejected samples
)

### Check the generated data

Check the acceptance rate, rejection reasons, class balance and commitment coverage before increasing the sample count. Rejected candidates have no valid eigenvalue target and are therefore omitted from the CSV; their reasons are returned separately in `failure_counts`.

In [ ]:
attempted_samples = len(dataset) + sum(failure_counts.values())
summary = pd.Series(
    {
        "attempted": attempted_samples,
        "accepted": len(dataset),
        "acceptance_rate": len(dataset) / attempted_samples,
        "stable_label_0": int((dataset["label"] == 0).sum()),
        "unstable_label_1": int((dataset["label"] == 1).sum()),
        "critical_real_min": dataset["critical_real"].min(),
        "critical_real_max": dataset["critical_real"].max(),
    },
    name="value",
)

all_commitments = pd.MultiIndex.from_tuples(
    [
        (0, 0, 1), (0, 1, 0), (0, 1, 1),
        (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1),
    ],
    names=["u_2", "u_3", "u_6"],
)
commitment_summary = (
    dataset.groupby(["u_2", "u_3", "u_6", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=all_commitments, fill_value=0)
    .reindex(columns=[0, 1], fill_value=0)
)
commitment_summary.columns = ["stable_0", "unstable_1"]

print("Sample summary")
display(summary)

print("\nRejection reasons")
display(pd.Series(failure_counts, name="rejected_samples"))

print("\nCommitment summary")
display(commitment_summary.reset_index())

print("\nFirst five rows of the dataset")
display(dataset.head())

### Define learning columns

The regression target is `critical_real`, the real part of the rightmost physical eigenvalue in $1/\mathrm{s}$. The load feature is the scalar `P_load_total_mw`; fixed base-load shares are used later to reconstruct the bus-level ANDES inputs. `label`, `critical_imag` and `slack_power_mw` remain diagnostics and are not neural-network inputs or targets.

In [ ]:
FEATURE_COLUMNS = [
    "u_2", "u_3", "u_6",  # Commitment variables
    "P_g2_mw", "P_g3_mw", "P_g6_mw",  # SG power variables
    "P_s8_mw", "P_load_total_mw",  # Solar power and total load variables
]
REGRESSION_TARGET = "critical_real"

assert not dataset[FEATURE_COLUMNS + [REGRESSION_TARGET]].isna().any().any()
display(dataset[FEATURE_COLUMNS + [REGRESSION_TARGET]].head())

## 3. Prepare one training dataset

For this simplified tutorial, use every accepted sample for training and do not create validation or test sets. Keep the three binary commitment features unchanged. All continuous features are active powers in MW, so divide them by the 100 MVA system base. The base is a scaling divisor, not an upper bound: normalized total load is approximately 1.79–3.13 in the sampled range. Consequently, all metrics below are in-sample training metrics and do not estimate generalization performance.

In [ ]:
training_data = nnf.prepare_regression_data(
    dataset=dataset,
    feature_columns=FEATURE_COLUMNS,
    target_column=REGRESSION_TARGET,
    system_base_mva=const.SYSTEM_BASE_MVA,
)

input_scaling = pd.DataFrame(
    {
        "feature": training_data.feature_names,
        "divide_by": training_data.feature_scale,
    }
)
print(f"Training tensor: {tuple(training_data.features.shape)}")
display(input_scaling)

## 4. Train the ReLU feedforward regressor

Use the architecture $8\rightarrow48\rightarrow24\rightarrow1$, with ReLU activations after the two hidden layers. The final linear output directly predicts the critical eigenvalue real part $\hat\alpha$ in $1/\mathrm{s}$, so no Sigmoid layer is used. Train all accepted samples with mean squared error. The larger network and 1500 epochs reduce the in-sample regression error while keeping the later NCET formulation small enough for this tutorial.

In [ ]:
torch.manual_seed(RANDOM_SEED)

# Define the fixed 8 -> 48 -> 24 -> 1 regressor explicitly.
# Its linear output is the predicted critical real part in 1/s.
regressor = torch.nn.Sequential(
    torch.nn.Linear(len(FEATURE_COLUMNS), 48),
    torch.nn.ReLU(),
    torch.nn.Linear(48, 24),
    torch.nn.ReLU(),
    torch.nn.Linear(24, 1),
)

# Optimizer settings.
EPOCHS = 1500
LEARNING_RATE = 3e-3
WEIGHT_DECAY = 1e-4

# MSE trains the network to reproduce the continuous eigenvalue target.
loss_function = torch.nn.MSELoss()
optimizer = torch.optim.Adam(
    regressor.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Full-batch training is sufficient for this small tutorial dataset.
loss_history = []
regressor.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    predicted_alpha = regressor(training_data.features)
    loss = loss_function(predicted_alpha, training_data.targets)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))

regressor.eval()
regressor

In [ ]:
training_metrics = nnf.regression_metrics(
    regressor, training_data, stability_margin=const.STABILITY_MARGIN
)
MODEL_PATH = const.OUTPUT_DIR / "critical_eigenvalue_regressor.pt"
nnf.save_regressor(MODEL_PATH, regressor, training_data)

display(pd.Series(training_metrics, name="training metric").round(4))
print(f"Saved regressor: {MODEL_PATH}")

figure, axis = plt.subplots(figsize=(6, 3.5))
axis.plot(loss_history)
axis.set(xlabel="Epoch", ylabel="MSE loss", title="Regressor training loss")
axis.grid(alpha=0.25)
figure.tight_layout()

## 5. Formulate the one-period UC problem

The tutorial uses a one-period UC: it ignores transmission flows, line limits and power losses. The fixed inputs are total available load $P_L^{\mathrm{total}}$ and available Solar power $P_s^{\mathrm{available}}$. The decisions are the three SG commitments $u_g$, their outputs $P_g$, the always-online Slack output $P_0$, and actual Solar injection $P_s$. No ramping, startup or shutdown variables are needed.

> Therefore, this is a much simplified version of the actual UC problem.

$$
\begin{aligned}
\min \quad & c_0P_0+c_g^TP_g+c_u^Tu_g+c_{\mathrm{curt}}(P_s^{\mathrm{available}}-P_s) \\
\text{s.t.} \quad & P_g^{\min}u_g \le P_g \le P_g^{\max}u_g, \\
& P_0^{\min} \le P_0 \le P_0^{\max}, \\
& 0 \le P_s \le P_s^{\mathrm{available}}, \\
& P_0+\mathbf{1}^TP_g+P_s=P_L^{\mathrm{total}}, \\
& \mathbf{1}^Tu_g \ge 1, \qquad u_g\in\{0,1\}^3.
\end{aligned}
$$

The final inequality excludes the all-off commitment, which was also excluded from the training data. The illustrative costs make the bus-6 SG cheapest, so the unconstrained UC tends to choose the feasible but often unstable commitment $(0,0,1)$. Bus 3 is the next-cheapest unit, while bus 2 is deliberately more expensive. This ordering makes the cost-versus-stability tradeoff visible in the tutorial.

The optimization problem is formulated by CVXPY and solved with the SCIPY solver.

In [ ]:
# Fixed operating profile supplied to the one-period UC. ANDES uses the
# full vector below, but the UC and NN use only its scalar total because
# the bus shares remain fixed at their base-case values.
LOAD_SCALE = 1.00
load_available_mw = const.BASE_LOAD_POWER_MW * LOAD_SCALE
solar_available_mw = 95.0

# Illustrative one-hour costs. Bus order is (2, 3, 6).
SLACK_ENERGY_COST = 30.0
SG_ENERGY_COST = [60.0, 34.0, 31.0]
SG_COMMITMENT_COST = [50.0, 30.0, 10.0]
SOLAR_CURTAILMENT_COST = 100.0

In [ ]:
# UC decision variables.
u_sg = cp.Variable(3, boolean=True, name="u_sg")
p_sg_mw = cp.Variable(3, nonneg=True, name="p_sg_mw")
p_slack_mw = cp.Variable(nonneg=True, name="p_slack_mw")
p_solar_mw = cp.Variable(nonneg=True, name="p_solar_mw")

# Obtain the minimum and maximum power limits for each SG from ANDES
sg_min_mw = list(const.SG_POWER_MIN_MW)
sg_max_mw = list(const.SG_POWER_MAX_MW)
total_load_mw = float(load_available_mw.sum())

# Formulate the constraints
uc_constraints = [
    cp.multiply(sg_min_mw, u_sg) <= p_sg_mw,
    p_sg_mw <= cp.multiply(sg_max_mw, u_sg),
    const.SLACK_POWER_MIN_MW <= p_slack_mw,
    p_slack_mw <= const.SLACK_POWER_MAX_MW,                    # Slack power must be within limits
    p_solar_mw <= solar_available_mw,                          # Solar power cannot exceed available amount
    p_slack_mw + cp.sum(p_sg_mw) + p_solar_mw == total_load_mw,  # Power balance
    cp.sum(u_sg) >= 1,  # At least one SG must be online
]

# Construct the objective function
uc_cost = (
    SLACK_ENERGY_COST * p_slack_mw
    + cp.sum(cp.multiply(SG_ENERGY_COST, p_sg_mw))
    + cp.sum(cp.multiply(SG_COMMITMENT_COST, u_sg))
    + SOLAR_CURTAILMENT_COST * (solar_available_mw - p_solar_mw)
)
uc_problem = cp.Problem(cp.Minimize(uc_cost), uc_constraints)

In [ ]:
# SCIPY uses SciPy's MILP solver and requires no commercial license.
uc_problem.solve(solver=cp.SCIPY)
if uc_problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
    raise RuntimeError(f"UC solve failed with status: {uc_problem.status}")

uc_solution = pd.Series(
    {
        "objective": uc_problem.value,
        "u_2": round(u_sg.value[0]),
        "u_3": round(u_sg.value[1]),
        "u_6": round(u_sg.value[2]),
        "P_g2_mw": p_sg_mw.value[0],
        "P_g3_mw": p_sg_mw.value[1],
        "P_g6_mw": p_sg_mw.value[2],
        "P_slack_mw": p_slack_mw.value,
        "P_solar_mw": p_solar_mw.value,
        "solar_curtailment_mw": solar_available_mw - p_solar_mw.value,
        "total_load_mw": total_load_mw,
    },
    name="baseline UC",
)
display(uc_solution.round(4))

## 6. Embed the trained network with NCET

Use NCET to convert the trained ReLU regressor into exact mixed-integer linear constraints. Binary commitments remain unchanged, while every MW input is divided by `SYSTEM_BASE_MVA`. The NCET input is linked to the UC variables in exactly the same `FEATURE_COLUMNS` order used for training. Let $\hat\alpha^{\mathrm{NN}}$ be the predicted critical eigenvalue real part and enforce

$$\hat\alpha^{\mathrm{NN}}\leq-\epsilon,$$

The physical stability boundary is close to zero. This tutorial uses the regression safety buffer $\epsilon=0.01\;\mathrm{s}^{-1}$. This is an example-specific engineering choice rather than a statistically calibrated guarantee.

NCET is used in four steps:

1. Construct lower and upper bounds for every **normalized** NN input. The 100-MVA base is only a scaling divisor; physical limits must still come from the UC model.
    The bounds are used to determine the range of the output of each layer for the Big-M method.
2. Call `form_milp` to encode every Linear and ReLU operation as exact CVXPY mixed-integer constraints.
3. Equate the NCET input variable to the normalized UC variables in `FEATURE_COLUMNS` order.
4. Add $\hat\alpha^{\mathrm{NN}}\le-\epsilon$ to select operating points whose predicted critical real part is safely negative.

In [ ]:
# Step 1: construct bounds for one normalized NN input sample.
# Bounds do not include a batch dimension. Their order must exactly match
# FEATURE_COLUMNS: commitments, SG powers, Solar power and total load.

# Commitment variables are binary, so their normalized range is [0, 1].
commitment_lower = np.zeros(3)
commitment_upper = np.ones(3)

# SG power can be zero when offline. Its upper bound comes from each SG's
# physical Pmax divided by the base MVA: 50 MW / 100 MVA = 0.5 pu.
sg_power_lower = np.zeros(3)
sg_power_upper = (
    np.asarray(const.SG_POWER_MAX_MW) / const.SYSTEM_BASE_MVA
)

# Solar can be curtailed to zero and cannot exceed the available 95 MW.
# Thus its normalized range is [0, 0.95], not automatically [0, 1].
solar_lower = np.array([0.0])
solar_upper = np.array([solar_available_mw / const.SYSTEM_BASE_MVA])

# Total load is fixed for this one-period UC, so its lower and upper
# bounds are identical. It equals 223.7/100 = 2.237 pu and is correctly
# greater than one; base-MVA scaling does not impose a physical limit.
fixed_total_load_pu = total_load_mw / const.SYSTEM_BASE_MVA
load_lower = np.array([fixed_total_load_pu])
load_upper = np.array([fixed_total_load_pu])

nn_lower_bound = np.concatenate(
    [commitment_lower, sg_power_lower, solar_lower, load_lower]
)
nn_upper_bound = np.concatenate(
    [commitment_upper, sg_power_upper, solar_upper, load_upper]
)

assert len(nn_lower_bound) == len(FEATURE_COLUMNS)
assert np.all(nn_lower_bound <= nn_upper_bound)

# Displaying the bounds makes it clear that normalization and physical
# limits are separate concepts. In particular, the normalized total-load
# bound is greater than one because total demand exceeds 100 MW.
nn_bounds = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "lower_pu": nn_lower_bound,
        "upper_pu": nn_upper_bound,
    }
)
display(nn_bounds)

In [ ]:
# Step 2: encode the trained Linear/ReLU network as exact MILP
# constraints. eval() fixes the model in inference mode. The reduced
# formulation creates binaries only for ReLUs whose sign is uncertain
# under the bounds above.
regressor.eval()
nn_encoding = form_milp(
    regressor,
    Bounds(lower=nn_lower_bound, upper=nn_upper_bound),
    relu_binary_mode="reduced",
)
print(nn_encoding.stats)

In [ ]:
# Step 3: build the normalized UC feature vector in the exact training
# column order:
# [u_2, u_3, u_6, P_g2, P_g3, P_g6, P_s8, P_load_total].
uc_feature_expression = cp.hstack(
    [
        u_sg,
        p_sg_mw / const.SYSTEM_BASE_MVA,
        p_solar_mw / const.SYSTEM_BASE_MVA,
        fixed_total_load_pu,
    ]
)
# NCET creates its own CVXPY input and output variables. The equality
# below connects that internal input to the actual UC decisions.
nn_input_variables = list(nn_encoding.inputs.values())
if len(nn_input_variables) != 1:
    raise ValueError("This tutorial expects one neural-network input.")
nn_input = nn_input_variables[0]
predicted_critical_real = nn_encoding.outputs[0][0]

In [ ]:
# Step 4: require a negative predicted critical real part. A regression
# margin of 0.01 1/s is the smallest tested value that passed the ANDES
# spot check for this UC profile. It is not a formal safety guarantee.
STABILITY_REGRESSION_MARGIN = 0.01

stability_constraints = (
    uc_constraints
    + list(nn_encoding.constraints)
    + [
        nn_input == uc_feature_expression, # The NN input variables should be linked with the UC variables and total load
        predicted_critical_real <= -STABILITY_REGRESSION_MARGIN, # stability constraint
    ]
)
# The original UC objective is unchanged; only the exact NN encoding and
# the stability inequality are added to form a larger MILP.
stability_problem = cp.Problem(cp.Minimize(uc_cost), stability_constraints)

# Solve
stability_problem.solve(
    solver=cp.SCIPY,
    canon_backend=cp.SCIPY_CANON_BACKEND,
)
if stability_problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
    raise RuntimeError(
        f"Stability-constrained UC failed: {stability_problem.status}"
    )

stability_solution = pd.Series(
    {
        "objective": stability_problem.value,
        "u_2": round(u_sg.value[0]),
        "u_3": round(u_sg.value[1]),
        "u_6": round(u_sg.value[2]),
        "P_g2_mw": p_sg_mw.value[0],
        "P_g3_mw": p_sg_mw.value[1],
        "P_g6_mw": p_sg_mw.value[2],
        "P_slack_mw": p_slack_mw.value,
        "P_solar_mw": p_solar_mw.value,
        "solar_curtailment_mw": solar_available_mw - p_solar_mw.value,
        "total_load_mw": total_load_mw,
    },
    name="stability-constrained UC",
)
display(stability_solution.round(4))
print(
    "NCET-predicted critical real part: "
    f"{float(predicted_critical_real.value):.6f} 1/s"
)

## 7. Evaluate baseline and stability-constrained UC

For the same selected load and solar profiles, compare:

1. baseline UC without the neural-network constraint; and
2. stability-constrained UC with the NCET formulation.

Report cost, commitment, dispatch, curtailment and the NN-predicted critical real part, then independently pass both optimized operating points through the ANDES AC power flow, dynamic initialization and eigenvalue analysis. This final ANDES calculation—not the NN prediction—is the physical verification.

In [ ]:
comparison_rows = []

for case_name, solution in {
    "Baseline UC": uc_solution,
    "Stability-constrained UC": stability_solution,
}.items():
    commitment = tuple(
        int(round(solution[f"u_{bus}"])) for bus in (2, 3, 6)
    )
    sg_dispatch = tuple(
        float(solution[f"P_g{bus}_mw"]) for bus in (2, 3, 6)
    )
    point = ss.OperatingPoint(
        sg_online=commitment,
        sg_power_mw=sg_dispatch,
        gfl_power_mw=float(solution["P_solar_mw"]),
        load_power_mw=tuple(load_available_mw),
    )

    # Recalculate the predicted critical real part directly in PyTorch.
    raw_features = np.asarray(
        [
            *commitment,
            *sg_dispatch,
            point.gfl_power_mw,
            sum(point.load_power_mw),
        ],
        dtype=np.float32,
    )
    normalized_features = raw_features / training_data.feature_scale
    with torch.no_grad():
        predicted_alpha = float(
            regressor(torch.tensor(normalized_features)).item()
        )

    # Independently verify the optimized point with ANDES.
    with ss.silence_andes_output():
        system = ss.build_system(point)
        realized_slack_mw = ss.solve_power_flow(system)
        andes_result = ss.calculate_small_signal(
            system, realized_slack_mw
        )

    comparison_rows.append(
        {
            "case": case_name,
            "objective": solution["objective"],
            "commitment": commitment,
            "SG dispatch MW": tuple(round(p, 3) for p in sg_dispatch),
            "Solar MW": solution["P_solar_mw"],
            "planned Slack MW": solution["P_slack_mw"],
            "ANDES Slack MW": andes_result.slack_power_mw,
            "predicted critical real": predicted_alpha,
            "predicted label": int(
                predicted_alpha > -const.STABILITY_MARGIN
            ),
            "ANDES critical real": andes_result.critical_real,
            "ANDES label": andes_result.label,
        }
    )

comparison = pd.DataFrame(comparison_rows).set_index("case")
display(comparison.round(4))